## Extended metrics and leave one product code out

Notebook 8 reported RMSE and PICP at 90 percent. The CA2 evaluation plan asked for more than that: MAE, RMSE and R squared per task each with bootstrap 95 percent confidence intervals, Expected Calibration Error on five bin reliability diagrams, and Prediction Interval Coverage Probability at 80, 90 and 95 percent nominal coverage with a sharpness penalty so that trivially wide intervals do not get free credit. This notebook computes all of them.

It also runs the leave one product code out protocol. That protocol turns out to be already computed. Because the folds are grouped by product code, every out of fold prediction comes from a model that never trained on that code, so the out of fold predictions are themselves leave one product code out results. The five eligible codes inside the cross validation are reported here and code 15, which was held out entirely, is reported in Notebook 12.

Nothing is trained here. The saved out of fold predictions from Notebook 8 are loaded and analysed, so this runs on the laptop in seconds.

In [1]:
import numpy as np
import pandas as pd
import json,hashlib
from pathlib import Path
from scipy.stats import norm

DATA=Path(r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets")
REPO=Path(r"C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis")
RESULTS=REPO/"results"

assert DATA.exists(),f"data folder not found: {DATA}"
assert RESULTS.exists(),f"results folder not found: {RESULTS}"
assert (RESULTS/"oof_predictions.npz").exists(),"oof_predictions.npz missing, rerun Notebook 8"

d=np.load(RESULTS/"oof_predictions.npz")
oof_pred=d["oof_pred"]
oof_std=d["oof_std"]
oof_mask=d["oof_mask"]
y_arr=d["y_arr"]
groups=d["groups"]

targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
cal=json.load(open(RESULTS/"mttrajnet_stratified.json"))["calibration"]
fa=json.load(open(RESULTS/"fold_assignment.json"))
fold_codes=[fa["folds"][f"fold_{i+1}"] for i in range(3)]

s_cal=np.zeros_like(oof_std)
for k in range(4):
    for f in range(3):
        te=np.isin(groups,fold_codes[f])
        s_cal[te,k]=oof_std[te,k]*cal[targets[k]]["scale_per_fold"][f]

print("loaded out of fold predictions from:",RESULTS)
print()
print("batches with an out of fold prediction:",int(oof_mask.sum()),"of",len(oof_mask))
print("codes covered:",len(np.unique(groups[oof_mask])),"| code 15 excluded:",15 not in np.unique(groups[oof_mask]))
print("calibrated std built using the per fold scales from Notebook 8:",s_cal[oof_mask].min()>0)

loaded out of fold predictions from: C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results

batches with an out of fold prediction: 941 of 1005
codes covered: 24 | code 15 excluded: True
calibrated std built using the per fold scales from Notebook 8: True


## Metrics per target with bootstrap confidence intervals

CA2 asked for MAE, RMSE and R squared per task, each with bootstrap 95 percent confidence intervals. The intervals are computed by resampling the 941 out of fold predictions with replacement 2000 times and taking the 2.5th and 97.5th percentiles. This is a sensible thing to report because a single RMSE number gives no sense of how much of it is noise, and with three folds the fold spread alone is a weak indicator.

R squared is computed against the mean of the out of fold set. A negative value means the model does worse than predicting that mean, which is a real possibility here and is worth reporting honestly rather than hiding.

In [2]:
def boot_ci(y,p,fn,n=2000,seed=42):
    rng=np.random.RandomState(seed)
    idx=np.arange(len(y))
    vals=[fn(y[i],p[i]) for i in (rng.choice(idx,len(idx),replace=True) for _ in range(n))]
    return float(np.percentile(vals,2.5)),float(np.percentile(vals,97.5))

rmse_fn=lambda t,p:float(np.sqrt(np.mean((t-p)**2)))
mae_fn=lambda t,p:float(np.mean(np.abs(t-p)))
r2_fn=lambda t,p:float(1-np.sum((t-p)**2)/np.sum((t-np.mean(t))**2))

m=oof_mask
metrics={}
print(f"{'target':<18}{'RMSE':<8}{'95% CI':<18}{'MAE':<8}{'95% CI':<18}{'R2':<8}{'95% CI'}")
for k in range(4):
    t=y_arr[m,k]
    p=oof_pred[m,k]
    rm=rmse_fn(t,p)
    ma=mae_fn(t,p)
    r2=r2_fn(t,p)
    rl,rh=boot_ci(t,p,rmse_fn)
    ml,mh=boot_ci(t,p,mae_fn)
    r2l,r2h=boot_ci(t,p,r2_fn)
    metrics[targets[k]]={"rmse":round(rm,3),"rmse_ci":[round(rl,3),round(rh,3)],
                         "mae":round(ma,3),"mae_ci":[round(ml,3),round(mh,3)],
                         "r2":round(r2,3),"r2_ci":[round(r2l,3),round(r2h,3)],
                         "n":int(m.sum())}
    print(f"{targets[k]:<18}{rm:<8.3f}[{rl:.3f}, {rh:.3f}]{'':<4}{ma:<8.3f}[{ml:.3f}, {mh:.3f}]{'':<4}{r2:<8.3f}[{r2l:.3f}, {r2h:.3f}]")

print()
print("check: RMSE here matches Notebook 8's mean of the fold RMSEs to within rounding")
print("  Notebook 8 reported 3.261 / 7.698 / 0.585 / 0.345")
print("  computed here      ",[metrics[t]["rmse"] for t in targets])

target            RMSE    95% CI            MAE     95% CI            R2      95% CI
dissolution_av    3.262   [3.109, 3.422]    2.564   [2.435, 2.692]    0.050   [0.001, 0.094]
tbl_av_hardness   7.701   [6.992, 8.446]    5.161   [4.814, 5.564]    0.252   [0.126, 0.349]
tbl_rsd_weight    0.601   [0.453, 0.760]    0.293   [0.263, 0.327]    -0.063  [-0.097, -0.032]
fct_tensile       0.349   [0.333, 0.365]    0.273   [0.260, 0.287]    0.114   [0.036, 0.185]

check: RMSE here matches Notebook 8's mean of the fold RMSEs to within rounding
  Notebook 8 reported 3.261 / 7.698 / 0.585 / 0.345
  computed here       [3.262, 7.701, 0.601, 0.349]


## Calibration across nominal coverage levels

Notebook 8 reported PICP at 90 percent only. CA2 asked for 80, 90 and 95 percent, a five bin Expected Calibration Error, and a sharpness measure. Sharpness matters because coverage on its own can be bought by making the intervals very wide, so an interval that always covers but is uselessly large should not be scored as well calibrated. Reporting mean interval width alongside coverage makes that trade off visible.

The scale used for each fold is the one Notebook 8 fitted on the other two folds, so no batch is being judged by a scale that saw it.

In [3]:
def ece_bins(t,p,s,n_bins=5):
    z=np.abs(t-p)/s
    levels=np.linspace(0.05,0.95,n_bins)
    e=0.0
    for c in levels:
        zc=norm.ppf(0.5+c/2)
        e+=abs(np.mean(z<=zc)-c)
    return e/len(levels)

cov={}
print(f"{'target':<18}{'PICP80':<10}{'PICP90':<10}{'PICP95':<10}{'ECE(5bin)':<12}{'mean width':<13}{'width/RMSE'}")
for k in range(4):
    t=y_arr[m,k]
    p=oof_pred[m,k]
    s=s_cal[m,k]
    row={}
    for lvl in [0.80,0.90,0.95]:
        z=norm.ppf(0.5+lvl/2)
        row[f"picp{int(lvl*100)}"]=round(float(np.mean((t>=p-z*s)&(t<=p+z*s))),3)
    w=float(np.mean(2*norm.ppf(0.95)*s))
    row["ece_5bin"]=round(float(ece_bins(t,p,s)),3)
    row["mean_width_90"]=round(w,3)
    row["width_over_rmse"]=round(w/metrics[targets[k]]["rmse"],2)
    cov[targets[k]]=row
    print(f"{targets[k]:<18}{row['picp80']:<10}{row['picp90']:<10}{row['picp95']:<10}{row['ece_5bin']:<12}{w:<13.3f}{row['width_over_rmse']}")

print()
print("nominal targets are 0.80, 0.90 and 0.95. Closer is better.")
print("width/RMSE near 3.3 is what a well scaled 90 percent interval looks like for a normal error")

target            PICP80    PICP90    PICP95    ECE(5bin)   mean width   width/RMSE
dissolution_av    0.77      0.857     0.897     0.024       10.838       3.32
tbl_av_hardness   0.846     0.932     0.957     0.031       25.397       3.3
tbl_rsd_weight    0.93      0.946     0.96      0.163       1.962        3.26
fct_tensile       0.79      0.841     0.865     0.034       1.187        3.4

nominal targets are 0.80, 0.90 and 0.95. Closer is better.
width/RMSE near 3.3 is what a well scaled 90 percent interval looks like for a normal error


## Leave one product code out

CA2 specified holding out each of the six product codes with at least 50 batches in turn, and reporting predictive and calibration metrics under that shift. That protocol is already satisfied by the out of fold predictions. The folds are grouped by product code, so for any code, the fold model that predicted it never saw a single batch of it during training. Each code's out of fold result is therefore a leave one product code out result.

Five of the six eligible codes sit inside the cross validation and are reported here. The sixth, code 15, was excluded from cross validation entirely and is reported separately in Notebook 12, where it is compared against a low cluster control.

Per code results are also given for the four high hardness codes that remain in the cross validation, since those are the ones that carry the regime.

In [4]:
codes=sorted(int(c) for c in np.unique(groups[m]))
high={4,8,20,24}
n_per=pd.Series(groups[m]).value_counts()
eligible=[c for c in codes if n_per[c]>=50]

loco={}
print("LEAVE ONE PRODUCT CODE OUT, from the out of fold predictions")
print("every code below was predicted by a fold model that never trained on it")
print()
print(f"{'code':<7}{'n':<6}{'meanHard':<10}{'dissol':<9}{'hardness':<10}{'wRSD':<8}{'tensile':<9}{'PICP90 hard':<13}{'note'}")
for c in codes:
    sel=(groups==c)&m
    r=[float(np.sqrt(np.mean((oof_pred[sel,k]-y_arr[sel,k])**2))) for k in range(4)]
    z=norm.ppf(0.95)
    pic=float(np.mean((y_arr[sel,1]>=oof_pred[sel,1]-z*s_cal[sel,1])&(y_arr[sel,1]<=oof_pred[sel,1]+z*s_cal[sel,1])))
    tag=[]
    if c in high: tag.append("HIGH cluster")
    if c in eligible: tag.append("CA2 eligible")
    loco[int(c)]={"n":int(sel.sum()),"mean_hardness":round(float(y_arr[sel,1].mean()),1),
                  "rmse":{targets[k]:round(r[k],3) for k in range(4)},
                  "picp90_hardness":round(pic,3),
                  "high_cluster":c in high,"ca2_eligible":c in eligible}
    print(f"{c:<7}{int(sel.sum()):<6}{y_arr[sel,1].mean():<10.1f}{r[0]:<9.3f}{r[1]:<10.3f}{r[2]:<8.3f}{r[3]:<9.3f}{pic:<13.3f}{', '.join(tag)}")

print()
print("CA2 eligible codes with >=50 batches:",eligible,"plus code 15 in Notebook 12")
print("high cluster codes still inside the cross validation:",sorted(high))

out={"analysis":"extended metrics and leave one product code out, computed from the Notebook 8 out of fold predictions",
     "note":"the folds are grouped by product code, so every out of fold prediction comes from a model that never trained on that code. the out of fold predictions are therefore leave one product code out results. no training performed. the calibrated std uses the per fold scale Notebook 8 fitted on the other two folds",
     "n_batches":int(m.sum()),"n_codes":len(codes),"ood_code_excluded":15,
     "pooled_metrics":metrics,
     "note_on_rmse":"pooled RMSE over all 941 batches. Notebook 8 reports the mean of the three fold RMSEs, which differs slightly when folds disagree; both are correct, they are different quantities",
     "coverage":cov,
     "ca2_eligible_codes":eligible,
     "leave_one_code_out":loco}
with open(RESULTS/"extended_metrics.json","w") as fh:
    json.dump(out,fh,indent=2)
print()
print("saved",RESULTS/"extended_metrics.json")

LEAVE ONE PRODUCT CODE OUT, from the out of fold predictions
every code below was predicted by a fold model that never trained on it

code   n     meanHard  dissol   hardness  wRSD    tensile  PICP90 hard  note
1      95    50.9      3.406    4.754     0.454   0.318    0.979        CA2 eligible
2      17    53.6      2.712    3.696     0.236   0.316    0.941        
3      16    52.1      3.213    3.932     0.382   0.262    1.000        
4      22    90.4      2.054    19.665    0.192   0.134    1.000        HIGH cluster
5      3     49.7      3.138    4.633     0.729   0.212    1.000        
6      2     52.0      5.915    5.251     0.144   0.524    1.000        
7      8     53.4      3.125    4.351     3.228   0.333    1.000        
8      2     90.5      4.199    34.109    0.097   0.233    1.000        HIGH cluster
9      4     48.2      2.001    4.626     0.278   0.140    1.000        
10     27    52.6      4.439    4.452     0.223   0.750    0.963        
11     15    38.9      